# Slater determinant → MPS: `mps_cpmc_new.py` against ITensorGaussianMPS.jl

Both codes implement the Fishman–White construction ([PRB 92, 075132](https://doi.org/10.1103/PhysRevB.92.075132)): diagonalize the correlation matrix block by block, turn every block eigenvector into a ladder of nearest-neighbour Givens rotations, and apply the inverse circuit to an occupation product state. [ITensorGaussianMPS.jl](https://github.com/ITensor/ITensorGaussianMPS.jl) (v0.1.16, `slater_determinant_to_mps`) is the reference here. It runs in-process through `juliacall`, and every result from either code is also checked against the exact determinant amplitudes $\langle n|\Phi\rangle=\det\Phi[\text{occupied rows}]$ on the full $2^L$ (or $4^L$) space.

SUMMARY_PLACEHOLDER

### One-time setup

The Julia side lives in `itensor_gmps_env/` (Project + Manifest: ITensors 0.9.32, ITensorMPS 0.4.2, ITensorGaussianMPS 0.1.16, PythonCall 0.9.36). On a new machine:

```bash
brew install juliaup && juliaup add release                        # Julia 1.13
cd trot/gmps/itensor_gmps_env
julia --project=. -e 'using Pkg; Pkg.Registry.add("General");
    Pkg.Registry.add(url="https://github.com/ITensor/ITensorRegistry"); Pkg.instantiate()'
~/.trot/bin/python -m pip install juliacall                          # 0.9.36, matches PythonCall
```

ITensorGaussianMPS is only in the ITensor registry, as its README says. `juliacall` has to be imported before JAX, and it has to be pointed at the real Julia binary, not at juliaup's launcher. The import cell below does both.

In [ ]:
from pathlib import Path

# ---- small test determinants, each with a dense 2^L reference
L_SMALL = 12
EPS = 1.0e-10                    # purity tolerance: ITensor eigval_cutoff = your occupation_tolerance
CHIS = (1, 2, 3, 4, 6, 8, 12, 16, 24, 32)
SEED = 7
# ---- ITensorGaussianMPS README example
L_README, NF_README = 20, 10
# ---- spinful comparison: up = open chain, down = chain + staggered potential
L_SPIN, N_UP, N_DN, STAGGER = 8, 4, 3, 0.7
# ---- walkers away from the plan reference
WALKER_CHI = 4                   # mps_cpmc_new.Config.walker_channel_chi
DELTAS = (1e-4, 1e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0)
SAMPLES = 6
# ---- Julia project with ITensorGaussianMPS
JULIA_ENV = Path("itensor_gmps_env").resolve()

In [ ]:
import itertools
import os
import subprocess
import sys

# juliacall first (before JAX), pointed at the real binary rather than juliaup's launcher
os.environ["PYTHON_JULIAPKG_EXE"] = subprocess.run(
    ["julia", "--startup-file=no", "-e", "print(joinpath(Sys.BINDIR, Base.julia_exename()))"],
    capture_output=True, text=True, check=True).stdout
os.environ["PYTHON_JULIAPKG_PROJECT"] = str(JULIA_ENV)
from juliacall import Main as jl

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))           # this folder: mps_cpmc_new.py
import mps_cpmc_new as mcn

jl.seval("""
using ITensors, ITensorMPS, ITensorGaussianMPS, LinearAlgebra

# Site tensors as dense (left bond, physical, right bond) arrays, with size-1 bonds at the ends.
function dense_tensors(ψ::MPS)
    s, L = siteinds(ψ), length(ψ)
    map(1:L) do j
        l = j > 1 ? linkind(ψ, j - 1) : nothing
        r = j < L ? linkind(ψ, j) : nothing
        A = Array(ψ[j], filter(!isnothing, [l, s[j], r])...)
        reshape(A, isnothing(l) ? 1 : dim(l), dim(s[j]), isnothing(r) ? 1 : dim(r))
    end
end

# Fishman-White circuit: mode occupations, and the Givens list as rows (i1, i2, c, s), 1-based sites.
function fw_circuit(Φ; eigval_cutoff=1e-8, maxblocksize=size(Φ, 1))
    ns, V = slater_determinant_to_gmps(Matrix{Float64}(Φ), size(Φ, 1); eigval_cutoff, maxblocksize)
    return ns, [Float64(getfield(g, f)) for g in V.rotations, f in (:i1, :i2, :c, :s)]
end

# Spinless determinant -> MPS on Fermion sites. eigval_cutoff and maxblocksize go to the circuit,
# maxdim and cutoff to ITensorMPS.apply.
function fw_mps(Φ; kwargs...)
    s = siteinds("Fermion", size(Φ, 1); conserve_qns=true)
    return dense_tensors(slater_determinant_to_mps(s, Matrix{Float64}(Φ); kwargs...))
end

# Spinful determinant -> MPS on Electron sites, local basis |0>, |up>, |dn>, |up dn>.
function fw_mps_electron(Φu, Φd; kwargs...)
    s = siteinds("Electron", size(Φu, 1); conserve_qns=true)
    return dense_tensors(slater_determinant_to_mps(s, Matrix{Float64}(Φu), Matrix{Float64}(Φd); kwargs...))
end
""")
print("Julia", jl.seval("string(VERSION)"), "| ITensorGaussianMPS", jl.seval("string(pkgversion(ITensorGaussianMPS))"),
      "| ITensorMPS", jl.seval("string(pkgversion(ITensorMPS))"), "| ITensors", jl.seval("string(pkgversion(ITensors))"))

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#8e5bd0", "#d4a20f", "#d6457a", "#3b9ab2", "#7a5230"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 14, "axes.labelsize": 10, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1.0, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2.0,
    "lines.solid_capstyle": "round", "legend.frameon": False, "font.size": 10,
})
LEGEND = dict(loc="upper left", bbox_to_anchor=(1.01, 1.0))

In [ ]:
def to_np(tensors):
    """Julia arrays -> list of numpy (Dl, d, Dr) arrays."""
    return [np.array(t) for t in tensors]


def dense(tensors):
    """Dense state of an open MPS, first site most significant."""
    psi = np.ones((1, 1))
    for A in tensors:
        A = np.asarray(A)
        psi = np.einsum("xa,apb->xpb", psi, A).reshape(-1, A.shape[-1])
    return psi.reshape(-1)


def bonds(tensors):
    return [np.shape(A)[2] for A in tensors][:-1]


def infidelity(a, b):
    return 1.0 - (a @ b) ** 2 / ((a @ a) * (b @ b))


def amp_error(v, exact):
    """Largest amplitude error after normalising v and fixing its global sign against exact."""
    v = v * np.sign(v @ exact) * np.linalg.norm(exact) / np.linalg.norm(v)
    return np.abs(v - exact).max()


def exact_spinless(C):
    """<n|Phi> = det C[occupied rows], in the basis c+_{i1}...c+_{iN}|0> with i1 < ... < iN."""
    L, N = C.shape
    occ = np.array(list(itertools.combinations(range(L), N)))
    psi = np.zeros(2 ** L)
    psi[(1 << (L - 1 - occ)).sum(1)] = np.linalg.det(C[occ])
    return psi


def exact_electron(Cu, Cd, interleave_sign=True):
    """Local index n_up + 2 n_dn per site, fermion order (1up, 1dn, 2up, 2dn, ...). Without
    interleave_sign: the order (all up)(all down) instead."""
    L = Cu.shape[0]
    bits = (np.arange(2 ** L)[:, None] >> (L - 1 - np.arange(L))) & 1       # [config, site]
    downs_left = np.cumsum(bits, axis=1) - bits                               # downs strictly left of a site
    amp = exact_spinless(Cu)[:, None] * exact_spinless(Cd)[None, :]           # [up config, down config]
    if interleave_sign:                                                       # each up passes those downs
        amp = amp * (-1.0) ** (bits[:, None, :] * downs_left[None, :, :]).sum(-1)
    index = (bits[:, None, :] + 2 * bits[None, :, :]) @ (4 ** (L - 1 - np.arange(L)))
    psi = np.zeros(4 ** L)
    psi[index.ravel()] = amp.ravel()
    return psi


def chain_orbitals(L, N, potential=0.0):
    return np.linalg.eigh(mcn.hopping_matrix(L, 1.0) + np.diag(np.broadcast_to(potential, L)))[1][:, :N].copy()


def n_gates(plan):
    return int(plan.block_sizes.sum() - len(plan.block_sizes))


rng = np.random.default_rng(SEED)
CASES = {
    "chain L=12 N=6": chain_orbitals(L_SMALL, 6),
    "chain L=12 N=5": chain_orbitals(L_SMALL, 5),
    "random L=12 N=5": np.linalg.qr(rng.standard_normal((L_SMALL, 5)))[0],
}
EXACT = {name: exact_spinless(C) for name, C in CASES.items()}

## 1. The two implementations, step by step

| step | ITensorGaussianMPS (`src/gmps.jl`) | `mps_cpmc_new.py` |
|---|---|---|
| correlation matrix | $\Lambda=\Phi^*\Phi^T$ | $P=UU^T$ (real) |
| block for site $i$ | `isolate_subblock_eig`: grow $\Lambda[i{:}i{+}b]$ from $b=1$ until an eigenvalue is within `eigval_cutoff` of 0 or 1, at most `maxblocksize` | `make_orbital_plan`: `adaptive` grows the same way with no cap; `rank_exact` uses $B=\min(N,L-N)+1$ for every walker; `maximal` uses the rest of the chain |
| which mode | the lowest-entropy eigenvector, occupied or empty | `adaptive`: occupied iff $\lambda_{\min}>1-\lambda_{\max}$. This is the same criterion |
| Givens ladder | `givens_rotations`: `givens(v, n, n+1)` from the bottom up | `_rotate_mode_to_front`: `atan2(v[j], v[j-1])` from the bottom up |
| a new determinant | full re-plan | `channel_angles`: block sizes and occupations stay frozen from the reference; only the eigenvector is recomputed |
| gate | $\begin{pmatrix}1&&&\\&c&s&\\&-s&c&\\&&&1\end{pmatrix}$ on $(\lvert 00\rangle,\lvert 01\rangle,\lvert 10\rangle,\lvert 11\rangle)$ | `gate_pair`: the same matrix |
| application | `apply(reverse(gates), product state; maxdim, cutoff)`: orthogonalize onto the gate, then a block-sparse SVD | `channel_mps`: `_move_centre` onto the gate, then `split_pair` with the kept count per sector frozen by `plan_bonds` |
| truncation | NDTensors `truncate!`: the `maxdim` largest $s^2$, but a pair within 0.1 % is never split. `cutoff` bounds the discarded weight | `plan_bonds`: exactly the `chi_max` largest $s$. `cutoff` is on $s/s_{\max}$ |
| spin | Electron sites: an interleaved $2L$ fermion circuit, merged pairwise | two $d=2$ channels, joined by `combine_channels` with sign $(-1)^{n_\alpha q_\beta}$ |
| overall sign | not tracked (normalised MPS) | `gauge` = det of the rotated occupied rows |

ITensor's `maxblocksize=b` allows blocks of $b+1$ sites, because it slices $\Lambda[i{:}i{+}b]$. So `maxblocksize` $=\min(N,L-N)$ is ITensor's counterpart of `rank_exact`.

## 2. Exact conversion (no bond truncation), spinless, L = 12

ITensor runs with `eigval_cutoff = EPS`, and your plans with `occupation_tolerance = EPS`. "amp err" is the largest amplitude error against $\det$. For your code the amplitude is `gauge × MPS`, taken as is. For ITensor it is normalised and sign-fixed, because ITensor does not track the overall sign.

In [ ]:
print(f"{'determinant':17s} {'code':19s} {'gates':>5s} {'max D':>5s} {'1-F vs det':>11s} {'amp err':>9s} {'1-F vs ITensor':>15s}")
BONDS = {}
for name, C in CASES.items():
    ex = EXACT[name]
    it = to_np(jl.fw_mps(C, eigval_cutoff=EPS))
    v_it = dense(it)
    gates_it = len(np.array(jl.fw_circuit(C, eigval_cutoff=EPS)[1]))
    BONDS[name] = {"ITensor": bonds(it)}
    print(f"{name:17s} {'ITensor':19s} {gates_it:5d} {max(bonds(it)):5d} {infidelity(v_it, ex):11.1e} {amp_error(v_it, ex):9.1e} {'':>15s}")
    for mode in ("adaptive", "rank_exact", "maximal"):
        plan = mcn.make_orbital_plan(C, mode, EPS)
        tensors, charges, gauge = mcn.channel_mps(C, plan)
        v = float(gauge) * dense(tensors)
        BONDS[name][mode] = bonds(tensors)
        print(f"{'':17s} {'yours ' + mode:19s} {n_gates(plan):5d} {max(bonds(tensors)):5d} "
              f"{infidelity(v, ex):11.1e} {np.abs(v - ex).max():9.1e} {infidelity(v, v_it):15.1e}")
print()
for name, row in BONDS.items():
    print(name)
    for code_, b in row.items():
        print(f"   {code_:11s} {b}")

EXACT_PLACEHOLDER

### Circuits

Next, gate by gate. ITensor's block sizes can be read off its Givens list, because each ladder runs down to its start site. A **tie** is a step whose block holds both an exactly pure empty mode and an exactly pure occupied mode. Either choice is exact, and round-off decides which one each code takes. `forced_plan` is `make_orbital_plan` with ITensor's block sizes and occupied/empty choices imposed, so the gate angles can be compared one to one.

In [ ]:
def ladders(G, L):
    """Block size per start site from ITensor's Givens list: each ladder runs down to its start site."""
    B, ladder = np.ones(L - 1, int), []
    for i1 in G[:, 0].astype(int) - 1:
        if ladder and i1 > ladder[-1]:
            B[ladder[-1]], ladder = len(ladder) + 1, []
        ladder.append(i1)
    if ladder:
        B[ladder[-1]] = len(ladder) + 1
    return B


def forced_plan(C, occupation, block_sizes):
    """make_orbital_plan with the block sizes and the occupied/empty choice of each step imposed."""
    U, references = np.array(C, float), []
    for k, B in enumerate(block_sizes):
        _, vectors = np.linalg.eigh((U @ U.T)[k:k + B, k:k + B])
        v = vectors[:, -1 if occupation[k] else 0]
        references.append(v.copy())
        mcn._rotate_mode_to_front(U, k, v)
    return mcn.OrbitalPlan(np.asarray(occupation), np.asarray(block_sizes), tuple(references), False)


def ties(C, plan, eps=EPS):
    """Steps whose block holds both a pure empty and a pure occupied mode, along the plan's own path."""
    U, out = np.array(C, float), []
    for k, B in enumerate(plan.block_sizes):
        w, vectors = np.linalg.eigh((U @ U.T)[k:k + B, k:k + B])
        out.append(bool(w[0] < eps and w[-1] > 1 - eps))
        mcn._rotate_mode_to_front(U, k, vectors[:, -1 if plan.occupation[k] else 0])
    return np.array(out)


FORCED = {}
for name, C in CASES.items():
    ns, G = map(np.array, jl.fw_circuit(C, eigval_cutoff=EPS))
    occ_it, blocks_it = np.rint(ns).astype(int), ladders(G, len(C))
    plan = mcn.make_orbital_plan(C, "adaptive", EPS)
    FORCED[name] = fplan = forced_plan(C, occ_it, blocks_it)
    angles, _ = mcn.channel_angles(C, fplan, xp=np)
    theta = np.array([float(t) for _, t in angles])
    same_sites = np.array_equal(np.array([p for p, _ in angles]) + 1, G[:, 0].astype(int))
    tie = ties(C, plan)
    print(f"{name}:  block sizes equal: {np.array_equal(plan.block_sizes, blocks_it)}   "
          f"gates {n_gates(plan)} vs {len(G)}")
    print(f"   occupations  yours   {plan.occupation}")
    print(f"                ITensor {occ_it}")
    print(f"   choices differ at steps {np.flatnonzero(plan.occupation[:-1] != occ_it[:-1]).tolist()};  "
          f"tie steps (yours) {np.flatnonzero(tie).tolist()}")
    print(f"   ITensor's choices forced: same gate sites {same_sites}, "
          f"max ||c|-|c_IT|| = {np.abs(np.abs(np.cos(theta)) - np.abs(G[:, 2])).max():.1e}, "
          f"max ||s|-|s_IT|| = {np.abs(np.abs(np.sin(theta)) - np.abs(G[:, 3])).max():.1e}, "
          f"gates with (c,s) of opposite sign {int(np.sum(np.cos(theta) * G[:, 2] < 0))}/{len(G)}")

CIRCUIT_PLACEHOLDER

## 3. ITensorGaussianMPS's own example: L = 20, N_f = 10 open chain

The README builds the free-fermion ground state with `maxblocksize = 4` (and the default `eigval_cutoff = 1e-8`). `examples/spinless_fermion.jl` and `examples/hubbard_1d_spin_conservation.jl` additionally use `eigval_cutoff = 1e-4`. Your code has no block cap; its `adaptive` tolerance plays the role of `eigval_cutoff`. $\Delta E=\langle H_0\rangle-E_0$ is computed from the dense $2^{20}$ state, with $E_0$ the sum of the 10 lowest orbital energies.

In [ ]:
def chain_energy(psi, L, t=1.0):
    """<psi|H0|psi>/<psi|psi> for open-chain hopping; adjacent hops carry no Jordan-Wigner sign."""
    idx, energy = np.arange(2 ** L), 0.0
    for i in range(L - 1):
        bi, bj = L - 1 - i, L - 2 - i
        hop = idx[((idx >> bi) & 1) != ((idx >> bj) & 1)]
        energy += -t * psi[hop] @ psi[hop ^ ((1 << bi) | (1 << bj))]
    return energy / (psi @ psi)


energies, orbitals = np.linalg.eigh(mcn.hopping_matrix(L_README, 1.0))
C20, E0 = orbitals[:, :NF_README].copy(), energies[:NF_README].sum()
ex20 = exact_spinless(C20)
print(f"E0 = {E0:.12f}\n")
print(f"{'code and settings':44s} {'gates':>5s} {'max D':>5s} {'1-F vs det':>11s} {'dE':>10s}")
for label, kw in [("ITensor README: maxblocksize=4", dict(maxblocksize=4)),
                  ("ITensor examples: maxblocksize=4, cutoff 1e-4", dict(maxblocksize=4, eigval_cutoff=1e-4)),
                  ("ITensor eigval_cutoff=1e-8, no block cap", dict()),
                  ("ITensor eigval_cutoff=1e-4, no block cap", dict(eigval_cutoff=1e-4))]:
    tensors = to_np(jl.fw_mps(C20, **kw))
    v = dense(tensors)
    print(f"{label:44s} {len(np.array(jl.fw_circuit(C20, **kw)[1])):5d} {max(bonds(tensors)):5d} "
          f"{infidelity(v, ex20):11.2e} {chain_energy(v, L_README) - E0:10.2e}")
for eps in (1e-8, 1e-4):
    plan = mcn.make_orbital_plan(C20, "adaptive", eps)
    tensors, _, _ = mcn.channel_mps(C20, plan)
    v = dense(tensors)
    print(f"{f'yours adaptive, tolerance {eps:.0e}':44s} {n_gates(plan):5d} {max(bonds(tensors)):5d} "
          f"{infidelity(v, ex20):11.2e} {chain_energy(v, L_README) - E0:10.2e}")

README_PLACEHOLDER

## 4. Truncated conversion: a fixed maximum bond χ

ITensor: `slater_determinant_to_mps(...; maxdim=χ)`. Yours: `channel_mps(C, plan, plan_bonds(C, plan, chi_max=χ))`, planned on the determinant itself. Both apply the gates in the same order, move the orthogonality centre onto every gate, and truncate the two-site split. Two cases: the half-filled chain, whose Schmidt spectra are degenerate by particle–hole symmetry, and a random determinant with no symmetry.

In [ ]:
def plan_bonds_with(C, orbital_plan, chi, rule):
    """plan_bonds' dry run with the kept-count rule passed in; returns a BondPlan for channel_mps."""
    tensors = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in orbital_plan.occupation]
    charges = [np.zeros(1, int)]
    for o in orbital_plan.occupation:
        charges.append(charges[-1] + int(o))
    angles, _ = mcn.channel_angles(np.asarray(C), orbital_plan, xp=np)
    kept_all, centre = [], None
    for site, theta in reversed(angles):
        centre = mcn._move_centre(tensors, charges, centre, site, xp=np)
        pair = mcn.gate_pair(tensors[site], tensors[site + 1], theta, xp=np)
        Dl, _, _, Dr = pair.shape
        matrix = pair.reshape(2 * Dl, 2 * Dr)
        svds = [np.linalg.svd(matrix[np.ix_(r, c)], full_matrices=False)
                for r, c, _ in mcn.sector_plan(charges[site], charges[site + 2]).sectors]
        kept = rule([s for _, s, _ in svds], chi)
        kept_all.append(kept)
        truncated = mcn.sector_plan(charges[site], charges[site + 2], kept)
        left = [u[:, :k] for (u, s, vh), k in zip(svds, kept) if k]
        right = [s[:k, None] * vh[:k] for (u, s, vh), k in zip(svds, kept) if k]
        left, right = mcn._assemble(left, right, truncated, xp=np)
        tensors[site], tensors[site + 1] = left.reshape(Dl, 2, -1), right.reshape(-1, 2, Dr)
        charges[site + 1] = truncated.middle_charges
        centre = site + 1
    return mcn.BondPlan(tuple(kept_all), tuple(charges), max(map(len, charges)), float("nan"))


def itensor_rule(singular_values, chi):
    """NDTensors.truncate! with maxdim only: keep the chi largest s^2, but if the last kept and the
    first dropped are within 0.1 %, raise the cut above both (a pair is never split)."""
    P = np.sort(np.concatenate(singular_values) ** 2)[::-1]
    n, cut = min(chi, len(P)), 0.0
    if n < len(P):
        cut = (P[n - 1] + P[n]) / 2
        if abs(P[n - 1] - P[n]) < 1e-3 * P[n - 1]:
            cut += 1e-3 * P[n - 1]
    return tuple(int(np.sum(s ** 2 > cut)) for s in singular_values)


TRUNC = {}
for name in ("chain L=12 N=6", "random L=12 N=5"):
    C, ex, fplan = CASES[name], EXACT[name], FORCED[name]
    plan = mcn.make_orbital_plan(C, "adaptive", EPS)
    for chi in CHIS:
        it = to_np(jl.fw_mps(C, eigval_cutoff=EPS, maxdim=chi))
        v_it = dense(it)
        v_yours = dense(mcn.channel_mps(C, plan, mcn.plan_bonds(C, plan, chi_max=chi))[0])
        v_forced = dense(mcn.channel_mps(C, fplan, mcn.plan_bonds(C, fplan, chi_max=chi))[0])
        v_rule = dense(mcn.channel_mps(C, fplan, plan_bonds_with(C, fplan, chi, itensor_rule))[0])
        TRUNC[name, chi] = dict(itensor=infidelity(v_it, ex), yours=infidelity(v_yours, ex),
                                forced=infidelity(v_forced, v_it), forced_rule=infidelity(v_rule, v_it),
                                itensor_bonds=bonds(it))

FLOOR = 1e-16
fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
for color, name in zip(PALETTE, ("chain L=12 N=6", "random L=12 N=5")):
    for key, ls, marker, who in (("itensor", "-", "o", "ITensor"), ("yours", "--", "s", "yours (adaptive)")):
        ax.plot(CHIS, [max(TRUNC[name, chi][key], FLOOR) for chi in CHIS], ls=ls, marker=marker, ms=6,
                color=color, mec=SURFACE, mew=1.5, label=f"{name}, {who}")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(CHIS, [str(c) for c in CHIS])
ax.set_xlabel("maximum bond dimension χ"); ax.set_ylabel("1 − F  against the exact determinant")
ax.set_title("Truncated conversion: infidelity against χ")
ax.legend(**LEGEND)
plt.show()

print(f"{'χ':>3s}   {'determinant':16s} {'1-F ITensor':>11s} {'1-F yours':>10s} {'yours/IT':>8s}   "
      f"{'1-F(IT, forced plan)':>21s} {'1-F(IT, forced plan + IT rule)':>31s}   ITensor bonds")
for name in ("chain L=12 N=6", "random L=12 N=5"):
    for chi in CHIS:
        r = TRUNC[name, chi]
        print(f"{chi:3d}   {name:16s} {r['itensor']:11.3e} {r['yours']:10.3e} {r['yours'] / max(r['itensor'], FLOOR):8.2f}   "
              f"{r['forced']:21.1e} {r['forced_rule']:31.1e}   {r['itensor_bonds']}")

TRUNC_PLACEHOLDER

## 5. Spinful determinant: Electron sites against `combine_channels`

Up spins: open-chain ground state with N_up = 4. Down spins: the same chain plus a staggered potential ±0.7, with N_dn = 3, so the two channels differ. L = 8, and the dense reference has $4^8$ amplitudes. ITensor builds an interleaved 16-site fermion circuit, whose up and down gates act on non-adjacent sites, and merges it into Electron sites. Your code converts each channel separately and interleaves them with the sign $(-1)^{n_\alpha q_\beta}$.

In [ ]:
Cu = chain_orbitals(L_SPIN, N_UP)
Cd = chain_orbitals(L_SPIN, N_DN, STAGGER * (-1.0) ** np.arange(L_SPIN))
ex_spin = exact_electron(Cu, Cd)
ex_blocked = exact_electron(Cu, Cd, interleave_sign=False)

it = to_np(jl.fw_mps_electron(Cu, Cd, eigval_cutoff=EPS))
v_it = dense(it)
plan_u, plan_d = (mcn.make_orbital_plan(C, "adaptive", EPS) for C in (Cu, Cd))
alpha, q_alpha, gauge_a = mcn.channel_mps(Cu, plan_u)
beta, q_beta, gauge_b = mcn.channel_mps(Cd, plan_d)
combined, _ = mcn.combine_channels(alpha, q_alpha, beta, q_beta)
v_yours = float(gauge_a * gauge_b) * dense(combined)

print(f"ITensor vs det (interleaved order):  1-F = {infidelity(v_it, ex_spin):.1e}, amp err = {amp_error(v_it, ex_spin):.1e}")
print(f"yours   vs det (interleaved order):  1-F = {infidelity(v_yours, ex_spin):.1e}, "
      f"amp err = {np.abs(v_yours - ex_spin).max():.1e}  (gauge x MPS, no normalisation)")
print(f"yours   vs ITensor:                  1-F = {infidelity(v_yours, v_it):.1e}")
print(f"check that the sign convention is tested: 1-F(det with interleave sign, det without) = {infidelity(ex_spin, ex_blocked):.3f}")
print()
print("bonds  ITensor, default apply     ", bonds(it))
print("       ITensor, cutoff=1e-14      ", bonds(to_np(jl.fw_mps_electron(Cu, Cd, eigval_cutoff=EPS, cutoff=1e-14))))
print("       yours (D_alpha x D_beta)   ", bonds(combined))

SPIN_PLACEHOLDER

## 6. Walkers away from the plan reference

In CPMC your code plans once, on a reference determinant, and every walker reuses the frozen block sizes, occupations and per-sector kept counts: `make_walker_ops` → `channel_mps(qa, plan_a, bond_a)`. That fixes the shapes for JAX. ITensor has no frozen plan and re-plans each determinant from scratch. Here the reference is the half-filled chain (L = 12, N = 6, the `rhf` plan reference) and the walkers are $C_w=\mathrm{qr}(C_\text{ref}+\delta\,G)$ with $G$ i.i.d. normal. Lines show the mean over SAMPLES walkers per δ, shading the min–max range. Everything uses χ = 4 per channel, the default `walker_channel_chi`.

In [ ]:
C_ref = CASES["chain L=12 N=6"]
plan_a = mcn.make_orbital_plan(C_ref, "adaptive", EPS)
plan_r = mcn.make_orbital_plan(C_ref, "rank_exact", EPS)
bonds_a = mcn.plan_bonds(C_ref, plan_a, chi_max=WALKER_CHI)
bonds_r = mcn.plan_bonds(C_ref, plan_r, chi_max=WALKER_CHI)

METHODS = {
    "itensor": "ITensor, re-planned per walker",
    "replanned": "yours, adaptive plan + bonds re-planned per walker",
    "frozen_adaptive": "yours, adaptive plan + bonds frozen on the reference (CPMC default)",
    "frozen_rank_exact": "yours, rank_exact plan + bonds frozen on the reference",
}
walk = {key: np.zeros((len(DELTAS), SAMPLES)) for key in (*METHODS, "plan_only", "distance")}
rng = np.random.default_rng(SEED)
for i, delta in enumerate(DELTAS):
    for j in range(SAMPLES):
        Cw = np.linalg.qr(C_ref + delta * rng.standard_normal(C_ref.shape))[0]
        ex = exact_spinless(Cw)
        plan_w = mcn.make_orbital_plan(Cw, "adaptive", EPS)
        walk["distance"][i, j] = 1 - np.linalg.det(C_ref.T @ Cw) ** 2
        walk["itensor"][i, j] = infidelity(dense(to_np(jl.fw_mps(Cw, eigval_cutoff=EPS, maxdim=WALKER_CHI))), ex)
        walk["replanned"][i, j] = infidelity(dense(mcn.channel_mps(Cw, plan_w, mcn.plan_bonds(Cw, plan_w, chi_max=WALKER_CHI))[0]), ex)
        walk["frozen_adaptive"][i, j] = infidelity(dense(mcn.channel_mps(Cw, plan_a, bonds_a)[0]), ex)
        walk["frozen_rank_exact"][i, j] = infidelity(dense(mcn.channel_mps(Cw, plan_r, bonds_r)[0]), ex)
        walk["plan_only"][i, j] = infidelity(dense(mcn.channel_mps(Cw, plan_a)[0]), ex)

fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
for color, (key, label) in zip(PALETTE, METHODS.items()):
    ax.fill_between(DELTAS, walk[key].min(1), walk[key].max(1), color=color, alpha=0.15, lw=0)
    ax.plot(DELTAS, walk[key].mean(1), marker="o", ms=6, color=color, mec=SURFACE, mew=1.5, label=label)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("walker perturbation δ"); ax.set_ylabel(f"1 − F  against the exact walker, χ = {WALKER_CHI}")
ax.set_title("Walkers away from the plan reference")
ax.legend(**LEGEND)
plt.show()

print(f"{'δ':>6s} {'1-|<ref|w>|^2':>13s} " + " ".join(f"{k:>18s}" for k in METHODS) + f" {'frozen/ITensor':>15s}")
for i, delta in enumerate(DELTAS):
    means = [walk[k][i].mean() for k in METHODS]
    print(f"{delta:6.0e} {walk['distance'][i].mean():13.2e} " + " ".join(f"{m:18.3e}" for m in means)
          + f" {means[2] / means[0]:15.2f}")
print(f"\nfrozen adaptive plan without bond truncation: max 1-F over all walkers = {walk['plan_only'].max():.1e}")

WALKER_PLACEHOLDER

## Conclusions

CONCLUSIONS_PLACEHOLDER